In [1]:
# Modules to install
# pip install scikit-clean pandas numpy sklearn

# Importo librerías

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
import warnings
import pickle


# My libraries
from filters import *
from noisers.classes import *
from noisers.funcs import *
from testFuncs import *
from noise_cv_eval import run_5cv_experiment, run_5cv_grid, run_5cv_baseline

rs = 33

/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_root = Path("dataset")
keel_datasets = sorted([p.name for p in (dataset_root / "data_base").iterdir() if p.is_dir()])
print(f"Available datasets: {keel_datasets}")


Available datasets: ['autos', 'balance', 'banana', 'car', 'cleveland', 'contraceptive', 'dermatology', 'ecoli', 'flare', 'german', 'glass', 'hayes-roth', 'heart', 'ionosphere', 'iris', 'led7digit', 'lymphography', 'magic', 'monk-2', 'newthyroid', 'nursery', 'page-blocks', 'penbased', 'phoneme', 'pima', 'ring', 'satimage', 'segment', 'shuttle', 'sonar', 'spambase', 'splice', 'thyroid', 'twonorm', 'vehicle', 'vowel', 'wdbc', 'wine', 'yeast', 'zoo']


### Display available filters

In [3]:
print_available_filters()


['AllKNN',
 'TomekLinks',
 'ENNFilter',
 'ENNProb',
 'ENNTh',
 'MultiEditFilter',
 'NCNEdit',
 'ClassificationFilter',
 'CVCFFilter',
 'EnsembleFiltering',
 'INFFCFilter',
 'IterativePartitioningFilter',
 'TABPFNClassificationFilter']

# Class random noise

In [13]:
def summarize_results(all_results):
    rows = []
    for res in all_results:
        dataset = res["dataset"]
        baseline_mean = (
            res["baseline_df"][["bal_acc", "f1_macro", "precision_macro", "recall_macro"]]
            .mean()
            .add_prefix("baseline_")
        )
        class_df = res["class_summary_df"].copy()
        class_df["method_base"] = class_df["method"].str.split("_", n=1).str[0]
        best_class = (
            class_df.sort_values(
                by=["method_base", "f1_macro_mean", "bal_acc_mean", "precision_macro_mean", "recall_macro_mean"],
                ascending=[True, False, False, False, False],
            )
            .groupby("method_base", as_index=False)
            .head(1)
        )
        removal_df = res["removal_summary_df"].copy()
        removal_df["method_base"] = removal_df["filter"].str.split("_", n=1).str[0]
        best_removal = (
            removal_df.merge(
                best_class[["dataset", "noise_type", "noise_pct", "seed", "k", "method", "method_base"]],
                left_on=["dataset", "noise_type", "noise_pct", "seed", "k", "method_base"],
                right_on=["dataset", "noise_type", "noise_pct", "seed", "k", "method_base"],
                how="inner",
                suffixes=("", "_bestclass"),
            )
        )
        for _, c_row in best_class.iterrows():
            m = c_row["method_base"]
            r_row = best_removal[best_removal["method_base"] == m].iloc[0] if not best_removal[best_removal["method_base"] == m].empty else None
            row = {
                "dataset": dataset,
                "method_base": m,
                **baseline_mean.to_dict(),
                "class_method": c_row["method"],
                "class_f1_macro_mean": c_row["f1_macro_mean"],
                "class_bal_acc_mean": c_row["bal_acc_mean"],
                "class_precision_macro_mean": c_row["precision_macro_mean"],
                "class_recall_macro_mean": c_row["recall_macro_mean"],
            }
            if r_row is not None:
                row.update({
                    "removal_filter": r_row["filter"],
                    "removal_f1_removal_mean": r_row["f1_removal_mean"],
                    "removal_mcc_mean": r_row["mcc_mean"],
                    "removal_recall_removal_mean": r_row["recall_removal_mean"],
                    "removal_precision_removal_mean": r_row["precision_removal_mean"],
                    "removal_acc_removal_mean": r_row["acc_removal_mean"],
                    "removal_specificity_mean": r_row["specificity_mean"],
                    "removal_removed_pct_mean": r_row["removed_pct_mean"],
                })
            rows.append(row)
    return pd.DataFrame(rows)

## Low noise (5%)

### Filtros basados en distancia

In [ ]:
from sklearn.linear_model import LogisticRegression

# Rejilla global de hiperparámetros
ks = [1, 3, 5, 9, 15]   # Número de vecinos
thresholds = [0.5, 0.75]  # Cota probabilística
n_blocks = [3, 5, 7, 9] 
max_iters = [1,3,5]

def build_distance_grid(ks, thresholds, n_blocks):
    '''
    Returns a list of filters, eaach with a different hyperparameter config
    '''
    distance_grid = []
    # ENN base 1-5
    for k in ks:
        distance_grid.append({
            "name": f"ENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="enn", n_jobs=-1),
        })

    # MENN 6-10
    for k in ks:
        distance_grid.append({
            "name": f"MENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="menn", n_jobs=-1),
        })

    # ENNProb y ENNProb+Th 11-25
    for k in ks:
        distance_grid.append({
            "name": f"ENNProb_k{k}",
            "filter": ENNProbFilter(n_neighbors=k, n_jobs=-1),
        }) 
        for tau in thresholds:
            distance_grid.append({
                "name": f"ENNTh_k{k}_tau{tau}",
                "filter": ENNProbFilter(n_neighbors=k, mode="th", threshold=tau, n_jobs=-1),
            })

    # #NCNEdit 26-30
    # for k in ks:
    #     distance_grid.append({
    #         "name": f"NCNEdit_k{k}",
    #         "filter": NCNEdit(n_neighbors=k, n_jobs=-1),
    #     })

    # Multiedit 30 - 90
    for k in ks:
        for b in n_blocks:
            for mi in max_iters:
                distance_grid.append({
                    "name": f"MultiEdit_k{k}_b{b}",
                    "filter": MultiEditFilter(n_neighbors=k, n_blocks=b, n_jobs=-1, max_iter=mi),
                })

    return distance_grid

# Create the distance_based_filter_list
distance_grid = build_distance_grid(ks, thresholds, n_blocks)

# Specify dataset info
noise_type = "cla_rand"
seed = 1
noise_k = 25
folds = (1, 2, 3, 4, 5)

all_results = []

# Test filters in each dataset
for dataset in keel_datasets:
    # Compute baseline (no filter) results
    baseline_df = run_5cv_baseline(
        dataset=dataset,
        noise_type=noise_type,
        seed=seed,
        k=noise_k,
        encoding="onehot",
        test_source="noisy",
        folds=folds,
    )

    # Compute results using filters
    distance_experiments_5 = [
        {
            "dataset": dataset,
            "noise_type": noise_type,
            "seed": seed,
            "k": noise_k,
            "filters": {cfg["name"]: cfg["filter"]},
            "encoding": "onehot",
            "test_source": "clean",
            "folds": folds,
            "summarize": True,
            "classifier": LogisticRegression(max_iter=1000),
            "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
        }
        for cfg in distance_grid
    ]
    classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
        distance_experiments_5,
        save_path="results_5cv",
        save_format="pickle",
        warnings_path="results_5cv/warnings_run_5cv_grid.txt",
        clear_warnings_file=False,
    )
    all_results.append(
        {
            "dataset": dataset,
            "baseline_df": baseline_df,
            "classification_df": classification_df,
            "removal_df": removal_df,
            "class_summary_df": class_summary_df,
            "removal_summary_df": removal_summary_df,
        }
    )
    with open(f"all_results_dists_{noise_type}_{seed}_{noise_k}.pkl", "wb") as f:
        pickle.dump(all_results,f)

5CV experiments: 100%|██████████| 85/85 [00:36<00:00,  2.30it/s]
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1248: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(classification_frames, ignore_index=True)
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1260: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_summary_df = pd.concat(class_summary_frames, ignore_index=True)
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1261: FutureWarning: The behavior o

KeyboardInterrupt: 

In [ ]:
x = summarize_results(all_results).drop(["dataset","class_method", "removal_filter"], axis=1).groupby("method_base").mean()
y = summarize_results(all_results).drop(["dataset","class_method", "removal_filter"], axis=1).groupby("method_base").mean()

,dataset,method_base,baseline_bal_acc,baseline_f1_macro,baseline_precision_macro,baseline_recall_macro,class_method,class_f1_macro_mean,class_bal_acc_mean,class_precision_macro_mean,class_recall_macro_mean,removal_filter,removal_f1_removal_mean,removal_mcc_mean,removal_recall_removal_mean,removal_precision_removal_mean,removal_acc_removal_mean,removal_specificity_mean,removal_removed_pct_mean
0,autos,ENN,0.797852,0.796910,0.830823,0.797852,ENN_k1,0.488432,0.511889,0.526454,0.498593,ENN_k1,0.238238,0.298982,1.000000,0.135444,0.679257,0.662263,37.105069
1,autos,ENNProb,0.797852,0.796910,0.830823,0.797852,ENNProb_k5,0.508374,0.533741,0.543663,0.519519,ENNProb_k1,0.238238,0.298982,1.000000,0.135444,0.679257,0.662263,37.105069
2,autos,ENNTh,0.797852,0.796910,0.830823,0.797852,ENNTh_k3_tau0.5,0.490143,0.523370,0.505928,0.509148,ENNTh_k1_tau0.5,0.238238,0.298982,1.000000,0.135444,0.679257,0.662263,37.105069
3,autos,MENN,0.797852,0.796910,0.830823,0.797852,MENN_k1,0.488432,0.511889,0.526454,0.498593,MENN_k1,0.238238,0.298982,1.000000,0.135444,0.679257,0.662263,37.105069
4,autos,MultiEdit,0.797852,0.796910,0.830823,0.797852,MultiEdit_k1_b9,0.475380,0.526222,0.492945,0.507481,MultiEdit_k1_b3,0.173610,0.208853,0.960000,0.095593,0.536159,0.513320,51.099902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,zoo,ENN,0.925397,0.903651,0.916667,0.925397,ENN_k5,0.801187,0.834524,0.802381,0.834524,ENN_k1,0.444444,0.454102,0.733333,0.323831,0.925741,0.932924,9.407407
130,zoo,ENNProb,0.925397,0.903651,0.916667,0.925397,ENNProb_k5,0.791262,0.820238,0.797143,0.820238,ENNProb_k1,0.444444,0.454102,0.733333,0.323831,0.925741,0.932924,9.407407
131,zoo,ENNTh,0.925397,0.903651,0.916667,0.925397,ENNTh_k1_tau0.5,0.778201,0.809524,0.790000,0.809524,ENNTh_k1_tau0.5,0.444444,0.454102,0.733333,0.323831,0.925741,0.932924,9.407407
132,zoo,MENN,0.925397,0.903651,0.916667,0.925397,MENN_k1,0.791758,0.823810,0.796349,0.823810,MENN_k1,0.499048,0.530769,0.900000,0.350260,0.925679,0.927661,10.401235


### Filtros basados en clasificadores

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from filters.classifier_based import (
    ClassificationFilter,
    CVCFFilter,
    EnsembleFiltering,
    INFFCFilter,
    IterativePartitioningFilter,
    TABPFNClassificationFilter,
    c45_like,
)

# Rejilla global de hiperparámetros
cvs = [3, 5, 7, 9]
thresholds = [0.5, 0.75]
n_partitions = [3, 5, 7, 9]
base_estimators = [
    ("DT", DecisionTreeClassifier(criterion="entropy", splitter="best", random_state=33)),
    ("RF", RandomForestClassifier(random_state=33, n_jobs=-1)),
    ("KNN1", KNeighborsClassifier(n_neighbors=1)),
    ("LR", LogisticRegression(max_iter=1000)),
]

def build_classifier_grid(cvs, thresholds, n_partitions):
    ''' 
    Return classifier_based_filter_list
    '''
    classifier_grid = []
    
    # CF
    for cv in cvs: # 16
        for est_name, est in base_estimators:
            classifier_grid.append({
                "name": f"CF_{est_name}_cv{cv}",
                "filter": ClassificationFilter(
                    estimator=est,
                    cv=cv,
                    action="remove",
                    random_state=33,
                ),
            })
    
    #CVCF y CVCFTh 
    for cv in cvs: # 28
        classifier_grid.append({
            "name": f"CVCF_cv{cv}",
            "filter": CVCFFilter(
                estimator=c45_like,
                cv=cv,
                vote_rule="threshold",
                threshold=0.5,
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"CVCFTh_cv{cv}_tau{tau}",
                "filter": CVCFFilter(
                    estimator=c45_like,
                    cv=cv,
                    vote_rule="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    #EF y EFTh
    for cv in cvs: #40
        classifier_grid.append({
            "name": f"Ensemble_cv{cv}",
            "filter": EnsembleFiltering(
                estimators=[est for _, est in base_estimators],
                cv=cv,
                mode="majority",
                threshold=0.5,
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"EnsembleTh_cv{cv}_tau{tau}",
                "filter": EnsembleFiltering(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    mode="majority",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })


    #INFFC e INFFCTh
    for cv in cvs: #52
        classifier_grid.append({
            "name": f"INFFC_cv{cv}",
            "filter": INFFCFilter(
                estimators=[est for _, est in base_estimators],
                cv=cv,
                decision_rule="majority",
                action="remove",
                max_iter=20,
                max_removed_frac=0.5,
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"INFFCTh_cv{cv}_tau{tau}",
                "filter": INFFCFilter(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    decision_rule="threshold",
                    threshold=tau,
                    action="remove",
                    max_iter=20,
                    max_removed_frac=0.5,
                    random_state=33,
                ),
            })
    
    #IPF
    for p in n_partitions: # 56
        classifier_grid.append({
            "name": f"IPF_p{p}",
            "filter": IterativePartitioningFilter(
                estimator=c45_like,
                n_partitions=p,
                vote_rule="majority",
                action="remove",
                p_stop=0.01,
                k_patience=3,
                max_iter=20,
                random_state=33,
            ),
        })


    # for cv in cvs: # 60
    #     classifier_grid.append({
    #         "name": f"TABPFN_cv{cv}",
    #         "filter": TABPFNClassificationFilter(
    #             cv=cv,
    #             action="remove",
    #             random_state=33,
    #         ),
    #     })
    return classifier_grid
    
classifier_grid = build_classifier_grid(cvs, thresholds, n_partitions)
noise_type = "cla_rand"
seed = 1
noise_k = 5
folds = (1, 2, 3, 4, 5)
all_results = []
for dataset in keel_datasets:
    baseline_df = run_5cv_baseline(
        dataset=dataset,
        noise_type=noise_type,
        seed=seed,
        k=noise_k,
        encoding="onehot",
        test_source="noisy",
        folds=folds,
    )
    classifier_experiments_5 = [
        {
            "dataset": dataset,
            "noise_type": noise_type,
            "seed": seed,
            "k": noise_k,
            "filters": {cfg["name"]: cfg["filter"]},
            "encoding": "onehot",
            "test_source": "clean",
            "folds": folds,
            "summarize": True,
            "classifier": LogisticRegression(max_iter=1000),
            "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
        }
        for cfg in classifier_grid
    ]
    classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
        classifier_experiments_5,
        save_path="results_5cv",
        save_format="pickle",
        warnings_path="results_5cv/warnings_run_5cv_grid_classif.txt",
        clear_warnings_file=False,
    )
    all_results.append({
        "dataset": dataset,
        "baseline_df": baseline_df,
        "classification_df": classification_df,
        "removal_df": removal_df,
        "class_summary_df": class_summary_df,
        "removal_summary_df": removal_summary_df,
    })
    with open(f"all_results_classif_{noise_type}_{seed}_{noise_k}.pkl", "wb") as f:
        pickle.dump(all_results,f)

5CV experiments: 100%|██████████| 56/56 [06:39<00:00,  7.14s/it]
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1248: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(classification_frames, ignore_index=True)
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1260: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_summary_df = pd.concat(class_summary_frames, ignore_index=True)
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1261: FutureWarning: The behavior o

In [42]:
summarize_results(all_results).drop(["dataset","class_method", "removal_filter"], axis=1).groupby("method_base").mean()

,baseline_bal_acc,baseline_f1_macro,baseline_precision_macro,baseline_recall_macro,class_f1_macro_mean,class_bal_acc_mean,class_precision_macro_mean,class_recall_macro_mean,removal_f1_removal_mean,removal_mcc_mean,removal_recall_removal_mean,removal_precision_removal_mean,removal_acc_removal_mean,removal_specificity_mean,removal_removed_pct_mean
method_base,,,,,,,,,,,,,,,
CVCF,0.786917,0.772127,0.77708,0.786917,0.693510,0.729706,0.699739,0.709274,0.477430,0.414504,0.122456,0.689697,0.842944,0.985075,3.877572
CVCFTh,0.786917,0.772127,0.77708,0.786917,0.695255,0.732484,0.700679,0.712051,0.477430,0.414504,0.122456,0.689697,0.842944,0.985075,3.877572
Classification,0.786917,0.772127,0.77708,0.786917,0.774012,0.789211,0.774827,0.789211,0.595693,0.531590,0.897543,0.456651,0.784640,0.760516,35.668503
Ensemble,0.888669,0.869506,0.87397,0.888669,0.829592,0.864246,0.821536,0.864246,0.768267,0.728872,0.972532,0.636338,0.891971,0.871403,28.365052
EnsembleTh,0.888669,0.869506,0.87397,0.888669,0.842785,0.875556,0.837143,0.875556,0.768267,0.728872,0.972532,0.636338,0.891971,0.871403,28.365052
INFFC,0.888669,0.869506,0.87397,0.888669,0.795592,0.838056,0.775833,0.838056,0.726698,0.686260,0.986128,0.577021,0.865048,0.835617,31.551108
INFFCTh,0.888669,0.869506,0.87397,0.888669,0.827139,0.861270,0.821429,0.861270,0.726698,0.686260,0.986128,0.577021,0.865048,0.835617,31.551108
IPF,0.888669,0.869506,0.87397,0.888669,0.790634,0.826349,0.780348,0.826349,0.706808,0.658519,0.935658,0.576072,0.858823,0.839376,30.628259


## Medium noise (25%)

### Filtros basados en distancia

In [ ]:
from sklearn.linear_model import LogisticRegression

# Rejilla global de hiperparámetros
ks = [1, 3, 5, 9, 15]   # Número de vecinos
thresholds = [0.5, 0.75]  # Cota probabilística
n_blocks = [3, 5, 7, 9] 
max_iters = [1,3,5]

def build_distance_grid(ks, thresholds, n_blocks):
    '''
    Returns a list of filters, eaach with a different hyperparameter config
    '''
    distance_grid = []
    # ENN base 1-5
    for k in ks:
        distance_grid.append({
            "name": f"ENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="enn", n_jobs=-1),
        })

    # MENN 6-10
    for k in ks:
        distance_grid.append({
            "name": f"MENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="menn", n_jobs=-1),
        })

    # ENNProb y ENNProb+Th 11-25
    for k in ks:
        distance_grid.append({
            "name": f"ENNProb_k{k}",
            "filter": ENNProbFilter(n_neighbors=k, n_jobs=-1),
        }) 
        for tau in thresholds:
            distance_grid.append({
                "name": f"ENNTh_k{k}_tau{tau}",
                "filter": ENNProbFilter(n_neighbors=k, mode="th", threshold=tau, n_jobs=-1),
            })

    # #NCNEdit 26-30
    # for k in ks:
    #     distance_grid.append({
    #         "name": f"NCNEdit_k{k}",
    #         "filter": NCNEdit(n_neighbors=k, n_jobs=-1),
    #     })

    # Multiedit 30 - 90
    for k in ks:
        for b in n_blocks:
            for mi in max_iters:
                distance_grid.append({
                    "name": f"MultiEdit_k{k}_b{b}",
                    "filter": MultiEditFilter(n_neighbors=k, n_blocks=b, n_jobs=-1, max_iter=mi),
                })

    return distance_grid

# Create the distance_based_filter_list
distance_grid = build_distance_grid(ks, thresholds, n_blocks)

# Specify dataset info
noise_type = "cla_rand"
seed = 1
noise_k = 25
folds = (1, 2, 3, 4, 5)

all_results = []

# Test filters in each dataset
for dataset in keel_datasets:
    # Compute baseline (no filter) results
    baseline_df = run_5cv_baseline(
        dataset=dataset,
        noise_type=noise_type,
        seed=seed,
        k=noise_k,
        encoding="onehot",
        test_source="noisy",
        folds=folds,
    )

    # Compute results using filters
    distance_experiments_5 = [
        {
            "dataset": dataset,
            "noise_type": noise_type,
            "seed": seed,
            "k": noise_k,
            "filters": {cfg["name"]: cfg["filter"]},
            "encoding": "onehot",
            "test_source": "clean",
            "folds": folds,
            "summarize": True,
            "classifier": LogisticRegression(max_iter=1000),
            "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
        }
        for cfg in distance_grid
    ]
    classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
        distance_experiments_5,
        save_path="results_5cv",
        save_format="pickle",
        warnings_path="results_5cv/warnings_run_5cv_grid.txt",
        clear_warnings_file=False,
    )
    all_results.append(
        {
            "dataset": dataset,
            "baseline_df": baseline_df,
            "classification_df": classification_df,
            "removal_df": removal_df,
            "class_summary_df": class_summary_df,
            "removal_summary_df": removal_summary_df,
        }
    )
    with open(f"all_results_dists_{noise_type}_{seed}_{noise_k}.pkl", "wb") as f:
        pickle.dump(all_results,f)

5CV experiments: 100%|██████████| 85/85 [00:38<00:00,  2.20it/s]
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1248: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(classification_frames, ignore_index=True)
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1260: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  class_summary_df = pd.concat(class_summary_frames, ignore_index=True)
/home/juamp/Documents/Master/TFM/scripts/noise_cv_eval.py:1261: FutureWarning: The behavior o

KeyboardInterrupt: 

### Filtros basados en clasificadores

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from filters.classifier_based import (
    ClassificationFilter,
    CVCFFilter,
    EnsembleFiltering,
    INFFCFilter,
    IterativePartitioningFilter,
    TABPFNClassificationFilter,
    c45_like,
)

# Rejilla global de hiperparámetros
cvs = [3, 5, 7, 9]
thresholds = [0.5, 0.75]
n_partitions = [3, 5, 7, 9]
base_estimators = [
    ("DT", DecisionTreeClassifier(criterion="entropy", splitter="best", random_state=33)),
    ("RF", RandomForestClassifier(random_state=33, n_jobs=-1)),
    ("KNN1", KNeighborsClassifier(n_neighbors=1)),
    ("LR", LogisticRegression(max_iter=1000)),
]

def build_classifier_grid(cvs, thresholds, n_partitions):
    ''' 
    Return classifier_based_filter_list
    '''
    classifier_grid = []
    
    # CF
    for cv in cvs: # 16
        for est_name, est in base_estimators:
            classifier_grid.append({
                "name": f"CF_{est_name}_cv{cv}",
                "filter": ClassificationFilter(
                    estimator=est,
                    cv=cv,
                    action="remove",
                    random_state=33,
                ),
            })
    
    #CVCF y CVCFTh 
    for cv in cvs: # 28
        classifier_grid.append({
            "name": f"CVCF_cv{cv}",
            "filter": CVCFFilter(
                estimator=c45_like,
                cv=cv,
                vote_rule="threshold",
                threshold=0.5,
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"CVCFTh_cv{cv}_tau{tau}",
                "filter": CVCFFilter(
                    estimator=c45_like,
                    cv=cv,
                    vote_rule="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    #EF y EFTh
    for cv in cvs: #40
        classifier_grid.append({
            "name": f"Ensemble_cv{cv}",
            "filter": EnsembleFiltering(
                estimators=[est for _, est in base_estimators],
                cv=cv,
                mode="majority",
                threshold=0.5,
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"EnsembleTh_cv{cv}_tau{tau}",
                "filter": EnsembleFiltering(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    mode="majority",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })


    #INFFC e INFFCTh
    for cv in cvs: #52
        classifier_grid.append({
            "name": f"INFFC_cv{cv}",
            "filter": INFFCFilter(
                estimators=[est for _, est in base_estimators],
                cv=cv,
                decision_rule="majority",
                action="remove",
                max_iter=20,
                max_removed_frac=0.5,
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"INFFCTh_cv{cv}_tau{tau}",
                "filter": INFFCFilter(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    decision_rule="threshold",
                    threshold=tau,
                    action="remove",
                    max_iter=20,
                    max_removed_frac=0.5,
                    random_state=33,
                ),
            })
    
    #IPF
    for p in n_partitions: # 56
        classifier_grid.append({
            "name": f"IPF_p{p}",
            "filter": IterativePartitioningFilter(
                estimator=c45_like,
                n_partitions=p,
                vote_rule="majority",
                action="remove",
                p_stop=0.01,
                k_patience=3,
                max_iter=20,
                random_state=33,
            ),
        })


    # for cv in cvs: # 60
    #     classifier_grid.append({
    #         "name": f"TABPFN_cv{cv}",
    #         "filter": TABPFNClassificationFilter(
    #             cv=cv,
    #             action="remove",
    #             random_state=33,
    #         ),
    #     })
    return classifier_grid
    
classifier_grid = build_classifier_grid(cvs, thresholds, n_partitions)
noise_type = "cla_rand"
seed = 1
noise_k = 25
folds = (1, 2, 3, 4, 5)
all_results = []
for dataset in keel_datasets:
    baseline_df = run_5cv_baseline(
        dataset=dataset,
        noise_type=noise_type,
        seed=seed,
        k=noise_k,
        encoding="onehot",
        test_source="noisy",
        folds=folds,
    )
    classifier_experiments_5 = [
        {
            "dataset": dataset,
            "noise_type": noise_type,
            "seed": seed,
            "k": noise_k,
            "filters": {cfg["name"]: cfg["filter"]},
            "encoding": "onehot",
            "test_source": "clean",
            "folds": folds,
            "summarize": True,
            "classifier": LogisticRegression(max_iter=1000),
            "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
        }
        for cfg in classifier_grid
    ]
    classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
        classifier_experiments_5,
        save_path="results_5cv",
        save_format="pickle",
        warnings_path="results_5cv/warnings_run_5cv_grid_classif.txt",
        clear_warnings_file=False,
    )
    all_results.append({
        "dataset": dataset,
        "baseline_df": baseline_df,
        "classification_df": classification_df,
        "removal_df": removal_df,
        "class_summary_df": class_summary_df,
        "removal_summary_df": removal_summary_df,
    })
    with open(f"all_results_classif_{noise_type}_{seed}_{noise_k}.pkl", "wb") as f:
        pickle.dump(all_results,f)

## High noise (50%)

### Filtros basados en distancia

In [ ]:
from sklearn.linear_model import LogisticRegression

# Rejilla global de hiperparámetros
ks = [1, 3, 5, 9, 15]   # Número de vecinos
thresholds = [0.5, 0.75]  # Cota probabilística
n_blocks = [3, 5, 7, 9] 
max_iters = [1,3,5]

def build_distance_grid(ks, thresholds, n_blocks):
    '''
    Returns a list of filters, eaach with a different hyperparameter config
    '''
    distance_grid = []
    # ENN base 1-5
    for k in ks:
        distance_grid.append({
            "name": f"ENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="enn", n_jobs=-1),
        })

    # MENN 6-10
    for k in ks:
        distance_grid.append({
            "name": f"MENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="menn", n_jobs=-1),
        })

    # ENNProb y ENNProb+Th 11-25
    for k in ks:
        distance_grid.append({
            "name": f"ENNProb_k{k}",
            "filter": ENNProbFilter(n_neighbors=k, n_jobs=-1),
        }) 
        for tau in thresholds:
            distance_grid.append({
                "name": f"ENNTh_k{k}_tau{tau}",
                "filter": ENNProbFilter(n_neighbors=k, mode="th", threshold=tau, n_jobs=-1),
            })

    # #NCNEdit 26-30
    # for k in ks:
    #     distance_grid.append({
    #         "name": f"NCNEdit_k{k}",
    #         "filter": NCNEdit(n_neighbors=k, n_jobs=-1),
    #     })

    # Multiedit 30 - 90
    for k in ks:
        for b in n_blocks:
            for mi in max_iters:
                distance_grid.append({
                    "name": f"MultiEdit_k{k}_b{b}",
                    "filter": MultiEditFilter(n_neighbors=k, n_blocks=b, n_jobs=-1, max_iter=mi),
                })

    return distance_grid

# Create the distance_based_filter_list
distance_grid = build_distance_grid(ks, thresholds, n_blocks)

# Specify dataset info
noise_type = "cla_rand"
seed = 1
noise_k = 50
folds = (1, 2, 3, 4, 5)

all_results = []

# Test filters in each dataset
for dataset in keel_datasets:
    # Compute baseline (no filter) results
    baseline_df = run_5cv_baseline(
        dataset=dataset,
        noise_type=noise_type,
        seed=seed,
        k=noise_k,
        encoding="onehot",
        test_source="noisy",
        folds=folds,
    )

    # Compute results using filters
    distance_experiments_5 = [
        {
            "dataset": dataset,
            "noise_type": noise_type,
            "seed": seed,
            "k": noise_k,
            "filters": {cfg["name"]: cfg["filter"]},
            "encoding": "onehot",
            "test_source": "clean",
            "folds": folds,
            "summarize": True,
            "classifier": LogisticRegression(max_iter=1000),
            "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
        }
        for cfg in distance_grid
    ]
    classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
        distance_experiments_5,
        save_path="results_5cv",
        save_format="pickle",
        warnings_path="results_5cv/warnings_run_5cv_grid.txt",
        clear_warnings_file=False,
    )
    all_results.append(
        {
            "dataset": dataset,
            "baseline_df": baseline_df,
            "classification_df": classification_df,
            "removal_df": removal_df,
            "class_summary_df": class_summary_df,
            "removal_summary_df": removal_summary_df,
        }
    )
    with open(f"all_results_dists_{noise_type}_{seed}_{noise_k}.pkl", "wb") as f:
        pickle.dump(all_results,f)

NameError: name 'ENNFilter' is not defined

### Filtros basados en clasificadores

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from filters.classifier_based import (
    ClassificationFilter,
    CVCFFilter,
    EnsembleFiltering,
    INFFCFilter,
    IterativePartitioningFilter,
    TABPFNClassificationFilter,
    c45_like,
)

# Rejilla global de hiperparámetros
cvs = [3, 5, 7, 9]
thresholds = [0.5, 0.75]
n_partitions = [3, 5, 7, 9]
base_estimators = [
    ("DT", DecisionTreeClassifier(criterion="entropy", splitter="best", random_state=33)),
    ("RF", RandomForestClassifier(random_state=33, n_jobs=-1)),
    ("KNN1", KNeighborsClassifier(n_neighbors=1)),
    ("LR", LogisticRegression(max_iter=1000)),
]

def build_classifier_grid(cvs, thresholds, n_partitions):
    ''' 
    Return classifier_based_filter_list
    '''
    classifier_grid = []
    
    # CF
    for cv in cvs: # 16
        for est_name, est in base_estimators:
            classifier_grid.append({
                "name": f"CF_{est_name}_cv{cv}",
                "filter": ClassificationFilter(
                    estimator=est,
                    cv=cv,
                    action="remove",
                    random_state=33,
                ),
            })
    
    #CVCF y CVCFTh 
    for cv in cvs: # 28
        classifier_grid.append({
            "name": f"CVCF_cv{cv}",
            "filter": CVCFFilter(
                estimator=c45_like,
                cv=cv,
                vote_rule="threshold",
                threshold=0.5,
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"CVCFTh_cv{cv}_tau{tau}",
                "filter": CVCFFilter(
                    estimator=c45_like,
                    cv=cv,
                    vote_rule="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    #EF y EFTh
    for cv in cvs: #40
        classifier_grid.append({
            "name": f"Ensemble_cv{cv}",
            "filter": EnsembleFiltering(
                estimators=[est for _, est in base_estimators],
                cv=cv,
                mode="majority",
                threshold=0.5,
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"EnsembleTh_cv{cv}_tau{tau}",
                "filter": EnsembleFiltering(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    mode="majority",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })


    #INFFC e INFFCTh
    for cv in cvs: #52
        classifier_grid.append({
            "name": f"INFFC_cv{cv}",
            "filter": INFFCFilter(
                estimators=[est for _, est in base_estimators],
                cv=cv,
                decision_rule="majority",
                action="remove",
                max_iter=20,
                max_removed_frac=0.5,
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"INFFCTh_cv{cv}_tau{tau}",
                "filter": INFFCFilter(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    decision_rule="threshold",
                    threshold=tau,
                    action="remove",
                    max_iter=20,
                    max_removed_frac=0.5,
                    random_state=33,
                ),
            })
    
    #IPF
    for p in n_partitions: # 56
        classifier_grid.append({
            "name": f"IPF_p{p}",
            "filter": IterativePartitioningFilter(
                estimator=c45_like,
                n_partitions=p,
                vote_rule="majority",
                action="remove",
                p_stop=0.01,
                k_patience=3,
                max_iter=20,
                random_state=33,
            ),
        })


    # for cv in cvs: # 60
    #     classifier_grid.append({
    #         "name": f"TABPFN_cv{cv}",
    #         "filter": TABPFNClassificationFilter(
    #             cv=cv,
    #             action="remove",
    #             random_state=33,
    #         ),
    #     })
    return classifier_grid
    
classifier_grid = build_classifier_grid(cvs, thresholds, n_partitions)
noise_type = "cla_rand"
seed = 1
noise_k = 50
folds = (1, 2, 3, 4, 5)
all_results = []
for dataset in keel_datasets:
    baseline_df = run_5cv_baseline(
        dataset=dataset,
        noise_type=noise_type,
        seed=seed,
        k=noise_k,
        encoding="onehot",
        test_source="noisy",
        folds=folds,
    )
    classifier_experiments_5 = [
        {
            "dataset": dataset,
            "noise_type": noise_type,
            "seed": seed,
            "k": noise_k,
            "filters": {cfg["name"]: cfg["filter"]},
            "encoding": "onehot",
            "test_source": "clean",
            "folds": folds,
            "summarize": True,
            "classifier": LogisticRegression(max_iter=1000),
            "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
        }
        for cfg in classifier_grid
    ]
    classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
        classifier_experiments_5,
        save_path="results_5cv",
        save_format="pickle",
        warnings_path="results_5cv/warnings_run_5cv_grid_classif.txt",
        clear_warnings_file=False,
    )
    all_results.append({
        "dataset": dataset,
        "baseline_df": baseline_df,
        "classification_df": classification_df,
        "removal_df": removal_df,
        "class_summary_df": class_summary_df,
        "removal_summary_df": removal_summary_df,
    })
    with open(f"all_results_classif_{noise_type}_{seed}_{noise_k}.pkl", "wb") as f:
        pickle.dump(all_results,f)

# TABPFN (individual)

### Low noise (5%)

In [ ]:
from filters import TABPFNClassificationFilter

tabpfn_classifier_filters = {
    "TABPFNClassificationFilter": TABPFNClassificationFilter(
        cv=5,
        action="remove",
        random_state=rs,
        tabpfn_params={"device": "auto"},
    ),
}

tabpfn_experiments_5 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 5,
        "filters": tabpfn_classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|5|tabpfn",
    }
    for dataset in keel_datasets
]

tabpfn_class_df_5, tabpfn_rem_df_5 = run_5cv_grid(tabpfn_experiments_5)
tabpfn_class_df_5


### Medium noise (25%)

In [ ]:
tabpfn_experiments_25 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": rs,
        "k": 25,
        "filters": tabpfn_classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|25|tabpfn",
    }
    for dataset in keel_datasets
]

tabpfn_class_df_25, tabpfn_rem_df_25 = run_5cv_grid(tabpfn_experiments_25)
tabpfn_class_df_25


### High noise (50%)

In [14]:
from filters import TABPFNClassificationFilter
tabpfn_classifier_filters = {
    "TABPFNClassificationFilter": TABPFNClassificationFilter(
        cv=5,
        action="remove",
        random_state=rs,
        tabpfn_params={"device": "auto"},
    ),
}

tabpfn_experiments_50 = [
    {
        "dataset": dataset,
        "noise_type": "cla_rand",
        "seed": 1,
        "k": 50,
        "filters": tabpfn_classifier_filters,
        "encoding": "onehot",
        "test_source": "clean",
        "folds": (1, 2, 3, 4, 5),
        "summarize": True,
        "experiment_name": f"{dataset}|cla_rand|50|tabpfn",
    }
    for dataset in ["iris"]
]

tabpfn_class_df_50, tabpfn_rem_df_50 = run_5cv_grid(tabpfn_experiments_50)
tabpfn_class_df_50


5CV experiments:   0%|          | 0/1 [00:57<?, ?it/s]


KeyboardInterrupt: 